In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode,hash,array
from pyspark.sql.types import ArrayType, IntegerType, ShortType

spark=SparkSession.builder.appName("negotiated_tare").config("spark.driver.memory","4g").getOrCreate()


25/04/29 17:41:24 WARN Utils: Your hostname, aayushgyawali resolves to a loopback address: 127.0.1.1; using 10.10.42.111 instead (on interface enp2s0)
25/04/29 17:41:24 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/29 17:41:25 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/29 17:41:26 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/04/29 17:41:26 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:

# df = spark.read.option("multiline",True).json("innetwork.json")
# _schema=billing_code string

In [3]:
df = spark.read.option("multiline",True).json("innetwork.json")
df.printSchema()

root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- billing_code_type_version: string (nullable = true)
 |-- description: string (nullable = true)
 |-- name: string (nullable = true)
 |-- negotiated_rates: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- negotiated_prices: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- additional_information: string (nullable = true)
 |    |    |    |    |-- billing_class: string (nullable = true)
 |    |    |    |    |-- billing_code_modifier: array (nullable = true)
 |    |    |    |    |    |-- element: string (containsNull = true)
 |    |    |    |    |-- expiration_date: string (nullable = true)
 |    |    |    |    |-- negotiated_rate: double (nullable = true)
 |    |    |    |    |-- negotiated_type: string (nullable = true)
 |    |    |    |    |-- service_code: array (nullable = true)
 |    |    |

In [4]:
from pyspark.sql.functions import explode, col


# First explode the negotiated_rates array
exploded_rates_df = df.withColumn("negotiated_rate", explode("negotiated_rates"))

# Then explode both provider_references and negotiated_prices while maintaining the relationship
rate_df = exploded_rates_df.withColumn("provider_group_id", explode("negotiated_rate.provider_references")) \
                           .withColumn("negotiated_price", explode("negotiated_rate.negotiated_prices")) \
                           .select(
                               col("billing_code"),
                               col("billing_code_type"),
                               col("negotiation_arrangement"),
                               col("negotiated_price.billing_class").alias("billing_class"),
                               col("negotiated_price.negotiated_rate").alias("negotiated_rate"),
                               col("negotiated_price.negotiated_type").alias("negotiated_type"),
                               col("provider_group_id"),
                               col("negotiated_price.billing_code_modifier").alias("billing_code_modifier"),
                               col("negotiated_price.service_code").alias("service_code"),
                            )

rate_df.show()

25/04/29 17:41:39 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors
                                                                                

+------------+-----------------+-----------------------+-------------+---------------+---------------+-----------------+---------------------+------------+
|billing_code|billing_code_type|negotiation_arrangement|billing_class|negotiated_rate|negotiated_type|provider_group_id|billing_code_modifier|service_code|
+------------+-----------------+-----------------------+-------------+---------------+---------------+-----------------+---------------------+------------+
|       PEMG1|              CPT|                    ffs| professional|           70.0|     negotiated|         10001001|                 NULL|        [11]|
|       PEMG1|              CPT|                    ffs| professional|           70.0|     negotiated|         10002001|                 NULL|        [11]|
|       PEMG1|              CPT|                    ffs| professional|           70.0|     negotiated|         10003001|                 NULL|        [11]|
|       PEMG1|              CPT|                    ffs| profess

## Hashing service_code

In [5]:
# hashing_df=rate_df.withColumn("service_code",hash("service_code"))
# hashing_df.show()
# hashing_df.printSchema()

## Rename

In [6]:

# rename_col = hashing_df.withColumnRenamed('billing_class','bCIs')\
#                 .withColumnRenamed('billing_code','bC')\
#                 .withColumnRenamed('billing_code_type','bCT')\
#                 .withColumnRenamed('negotiated_rate','negR')\
#                 .withColumnRenamed('negotiated_type','negT')\
#                 .withColumnRenamed('negotiation_arrangement','negA')\
#                 .withColumnRenamed('billing_code_modifier','mdH')\
#                 .withColumnRenamed('service_code','poSH')

In [7]:
rate_cast = rate_df.withColumn("service_code",col("service_code").cast(ArrayType(IntegerType())))
rate_cast.printSchema()


root
 |-- billing_code: string (nullable = true)
 |-- billing_code_type: string (nullable = true)
 |-- negotiation_arrangement: string (nullable = true)
 |-- billing_class: string (nullable = true)
 |-- negotiated_rate: double (nullable = true)
 |-- negotiated_type: string (nullable = true)
 |-- provider_group_id: long (nullable = true)
 |-- billing_code_modifier: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- service_code: array (nullable = true)
 |    |-- element: integer (containsNull = true)



In [8]:
# df_cast.toPandas().to_json('output2.json', orient='records')


In [13]:
# df_cast.write.parquet("innetwork.parquet")
rate_cast.coalesce(1).write.mode("overwrite").parquet("innetwork.parquet")

In [10]:
# dfa = spark.read.parquet("innetwork.parquet")
# dfa.show(10)

SyntaxError: invalid syntax (473536216.py, line 1)